In [1]:
import torch.nn as nn
import torch

In [2]:
torch.manual_seed(1)

rnn_layer = nn.RNN(input_size=5, hidden_size=2, num_layers=1, batch_first=True)

In [3]:
w_xh = rnn_layer.weight_ih_l0
w_hh = rnn_layer.weight_hh_l0
b_xh = rnn_layer.bias_ih_l0
b_hh = rnn_layer.bias_hh_l0

print(f'W_xh shape: {w_xh.shape}')
print(f'W_hh shape: {w_hh.shape}')
print(f'b_xh shape: {b_xh.shape}')
print(f'b_hh shape: {b_hh.shape}')

W_xh shape: torch.Size([2, 5])
W_hh shape: torch.Size([2, 2])
b_xh shape: torch.Size([2])
b_hh shape: torch.Size([2])


In [4]:
x_seq = torch.tensor([[1.0]*5, [2.0]*5, [3.0]*5])
x_seq.shape


torch.Size([3, 5])

In [5]:
output, hn = rnn_layer(torch.reshape(x_seq, (1, 3, 5)))

out_man = []

for t in range(3):

    xt = torch.reshape(x_seq[t], (1, 5))
    print(f'Time step {t} => ')
    print(f'     Input      : {xt.numpy()}')
    ht = torch.matmul(xt, torch.transpose(w_xh, 0, 1)) + b_xh

    print(f'     Hidden:  {ht.detach().numpy()}')

    if t > 0:
        prev_h = out_man[t-1]
    else:
        prev_h = torch.zeros((ht.shape))

    ot = ht + torch.matmul(prev_h, torch.transpose(w_hh, 0, 1)) + b_hh

    ot = torch.tanh(ot)
    out_man.append(ot)
    print(f'    Output manual: {ot.detach().numpy()}')
    print(f'    RNN output:  {output[:, t].detach().numpy()}')
    print('--------------------------------------------')

Time step 0 => 
     Input      : [[1. 1. 1. 1. 1.]]
     Hidden:  [[-0.47019297  0.58639044]]
    Output manual: [[-0.35198015  0.52525216]]
    RNN output:  [[-0.3519801   0.52525216]]
--------------------------------------------
Time step 1 => 
     Input      : [[2. 2. 2. 2. 2.]]
     Hidden:  [[-0.8888316  1.2364398]]
    Output manual: [[-0.68424344  0.76074266]]
    RNN output:  [[-0.68424344  0.76074266]]
--------------------------------------------
Time step 2 => 
     Input      : [[3. 3. 3. 3. 3.]]
     Hidden:  [[-1.3074702  1.8864892]]
    Output manual: [[-0.8649416  0.9046636]]
    RNN output:  [[-0.8649416  0.9046636]]
--------------------------------------------


In [6]:
from pathlib import Path

def load_acl_imdb(split_dir: Path):
    """Load aclImdb split into (label, text) tuples."""
    samples = []
    for label in ("pos", "neg"):
        label_dir = split_dir / label
        for file_path in label_dir.glob("*.txt"):
            text = file_path.read_text(encoding="utf-8", errors="ignore")
            samples.append((label, text))
    return samples

imdb_root = Path("../ch7-sentiment-analysis/aclImdb")
if not imdb_root.exists():
    raise FileNotFoundError(f"Could not find dataset at {imdb_root.resolve()}")

train_dataset = load_acl_imdb(imdb_root / "train")
test_dataset = load_acl_imdb(imdb_root / "test")

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(train_dataset[0][0], train_dataset[0][1][:120], "...")

Train samples: 25000
Test samples: 25000
pos For a movie that gets no respect there sure are a lot of memorable quotes listed for this gem. Imagine a movie where Joe ...


In [7]:
import warnings

# Silence torchdata/torchtext deprecation warnings in this educational notebook
warnings.filterwarnings("ignore", category=UserWarning, module=r"torchdata\\.datapipes")

import torchtext
torchtext.disable_torchtext_deprecation_warning()

from torchtext.datasets import IMDB

train_dataset = IMDB(split='train')
test_dataset = IMDB(split='test')

/Users/blaiseforgwa/projects/pyml-book/.venv/lib/python3.12/site-packages/torchdata/datapipes/__init__.py:18: UserWarning: 
################################################################################
WARNING!
The 'datapipes', 'dataloader2' modules are deprecated and will be removed in a
future torchdata release! Please see https://github.com/pytorch/data/issues/1196
to learn more and leave feedback.
################################################################################

  deprecation_warning()


In [8]:
from torch.utils.data import random_split

In [10]:
torch.manual_seed(1)
train_dataset, valid_dataset = random_split(
    list(train_dataset), [20000, 5000]
)

In [11]:
import re 
from collections import OrderedDict, Counter

def tokenizer(text):
    text = re.sub(r'<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text.lower())
    text = re.sub(r'[\W]+', ' ', text.lower()) +\
        ' '.join(emoticons).replace('-','')
    tokenized = text.split()

    return tokenized

In [13]:
token_counts = Counter()
for label, line in train_dataset:
    tokens = tokenizer(line)
    token_counts.update(tokens)

print(f'Vocab-size: {len(token_counts)}')

Vocab-size: 69023


In [14]:
import torchtext
torchtext.disable_torchtext_deprecation_warning()

from torchtext.vocab import vocab

sorted_by_freq_tuples = sorted(
    token_counts.items(), key=lambda x: x[1], reverse=True
)

In [15]:
ordered_dict = OrderedDict(sorted_by_freq_tuples)
vocab = vocab(ordered_dict)
vocab.insert_token("<pad>", 0)
vocab.insert_token("<unk>", 1)
vocab.set_default_index(1)


In [16]:
print([vocab[token] for token in ['this', 'is', 'an', 'example']])

[11, 7, 35, 457]


In [17]:
text_pipeline = lambda x: [vocab[token] for token in tokenizer(x)]
label_pipeline = lambda x:  1. if x == 'pos' else 0.

In [18]:
def collate_batch(batch):

    label_list, text_list, lengths = [], [], []

    for _label, _text in batch:
        label_list.append(label_pipeline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
        lengths.append(processed_text.size(0))
    
    label_list = torch.tensor(label_list)
    lengths = torch.tensor(lengths)
    padded_text_list = nn.utils.rnn.pad_sequence(
        text_list, batch_first=True
    )
    return padded_text_list, label_list, lengths

In [19]:
from torch.utils.data import DataLoader
dataloader = DataLoader(train_dataset, batch_size=4, shuffle=False, collate_fn=collate_batch)

In [20]:
text_batch, label_batch , length_batch = next(iter(dataloader))
print(text_batch)
print(label_batch)
print(length_batch)

tensor([[   35,  1739,     7,   449,   721,     6,   301,     4,   787,     9,
             4,    18,    44,     2,  1705,  2460,   186,    25,     7,    24,
           100,  1874,  1739,    25,     7, 34415,  3568,  1103,  7517,   787,
             5,     2,  4991, 12401,    36,     7,   148,   111,   939,     6,
         11598,     2,   172,   135,    62,    25,  3199,  1602,     3,   928,
          1500,     9,     6,  4601,     2,   155,    36,    14,   274,     4,
         42945,     9,  4991,     3,    14, 10296,    34,  3568,     8,    51,
           148,    30,     2,    58,    16,    11,  1893,   125,     6,   420,
          1214,    27, 14542,   940,    11,     7,    29,   951,    18,    17,
         15994,   459,    34,  2480, 15211,  3713,     2,   840,  3200,     9,
          3568,    13,   107,     9,   175,    94,    25,    51, 10297,  1796,
            27,   712,    16,     2,   220,    17,     4,    54,   722,   238,
           395,     2,   787,    32,    27,  5236,  

In [21]:
batch_size = 32
train_dl = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
valid_dl = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

In [23]:
embedding = nn.Embedding(
    num_embeddings=10,
    embedding_dim=3,
    padding_idx=0
)

text_encoded_input = torch.LongTensor([[1,2,4,5], [4,3,2,0]])
embedded_text = embedding(text_encoded_input)

print(f'Embeddings shape: {embedded_text.shape}')

print(embedded_text)

Embeddings shape: torch.Size([2, 4, 3])
tensor([[[-0.3098,  0.1645,  0.3430],
         [-0.5329, -0.7423,  0.2471],
         [ 1.9600, -0.3665,  0.1639],
         [ 0.1837,  2.7972,  1.0885]],

        [[ 1.9600, -0.3665,  0.1639],
         [-1.1142, -0.5028,  0.4700],
         [-0.5329, -0.7423,  0.2471],
         [ 0.0000,  0.0000,  0.0000]]], grad_fn=<EmbeddingBackward0>)


In [25]:
class RNN(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.rnn = nn.RNN(input_size, hidden_size, num_layers=2, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)


    def forward(self, x):
        _, hidden = self.rnn(x)
        out = hidden[-1, :, :] # use the final hidden state

        out = self.fc(out)

        return out
    

model = RNN(64, 32)
print(model)

model(torch.rand(5, 3, 64))


RNN(
  (rnn): RNN(64, 32, num_layers=2, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


tensor([[-0.1598],
        [-0.1391],
        [-0.0648],
        [ 0.0562],
        [-0.2274]], grad_fn=<AddmmBackward0>)

In [27]:
class RNN(nn.Module):

    def __init__(self, vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True)
        self.fc1 = nn.Linear(rnn_hidden_size, fc_hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(fc_hidden_size, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text, lengths):

        out = self.embedding(text)
        out = nn.utils.rnn.pack_padded_sequence(
            out, lengths.cpu().numpy(), enforce_sorted=False, batch_first=True
        )

        out, (hidden, cell) = self.rnn(out)
        out = hidden[-1, :, :]
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out



In [28]:
vocab_size = len(vocab)
embed_dim = 20
rnn_hidden_size = 64
fc_hidden_size = 64
torch.manual_seed(1)
model = RNN(vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size)

model

RNN(
  (embedding): Embedding(69025, 20, padding_idx=0)
  (rnn): LSTM(20, 64, batch_first=True)
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [37]:
def train(dataloader):

    model.train()

    total_acc, total_loss = 0, 0
    for text_batch, label_batch, lengths in dataloader:

        optimizer.zero_grad()
        pred = model(text_batch, lengths)[:, 0]
        loss = loss_fn(pred, label_batch)
        loss.backward()
        optimizer.step()

        total_acc += ((pred >= 0.5).float() == label_batch).float().sum().item()

        total_loss += loss.item()*label_batch.size(0)

    return total_acc/len(dataloader.dataset), total_loss/len(dataloader.dataset)

In [38]:
def evaluate(dataloader):
    model.eval()

    total_acc, total_loss = 0, 0
    with torch.inference_mode():

        for text_batch, label_batch, lengths in dataloader:
            pred = model(text_batch, lengths)[:, 0]
            loss = loss_fn(pred, label_batch)
            total_acc += (
                (pred >= 0.5).float() == label_batch
            ).float().sum().item()

            total_loss += loss.item()*label_batch.size(0)

    return total_acc/len(dataloader.dataset), total_loss/len(dataloader.dataset)

In [39]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [40]:
num_epochs = 10
torch.manual_seed(1)
for epoch in range(num_epochs):

    acc_train, loss_train = train(train_dl)
    acc_valid, loss_valid = evaluate(valid_dl)

    print(f'Epoch {epoch} accuracy: {acc_train:.4f}'
          f'  val accuracy: {acc_valid:.4f}')

Epoch 0 accuracy: 1.0000  val accuracy: 1.0000
Epoch 1 accuracy: 1.0000  val accuracy: 1.0000
Epoch 2 accuracy: 1.0000  val accuracy: 1.0000
Epoch 3 accuracy: 1.0000  val accuracy: 1.0000
Epoch 4 accuracy: 1.0000  val accuracy: 1.0000
Epoch 5 accuracy: 1.0000  val accuracy: 1.0000
Epoch 6 accuracy: 1.0000  val accuracy: 1.0000
Epoch 7 accuracy: 1.0000  val accuracy: 1.0000
Epoch 8 accuracy: 1.0000  val accuracy: 1.0000
Epoch 9 accuracy: 1.0000  val accuracy: 1.0000


In [42]:
class RNN(nn.Module):

    def __init__(self, vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )

        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(rnn_hidden_size*2, fc_hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(fc_hidden_size, 1)
        self.sigmoid = nn.Sigmoid()



    def forward(self, text, lengths):

        out =  self.embedding(text)
        out = nn.utils.rnn.pack_padded_sequence(
            out, lengths.cpu().numpy(), enforce_sorted=False, batch_first=True
        )

        _, (hidden, cell) = self.rnn(out)
        out = torch.concat(
            [hidden[-2, :, :], hidden[-1, :, :]], dim=1
        )

        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)

        return out

In [43]:
model = RNN(vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size)
model

RNN(
  (embedding): Embedding(69025, 20, padding_idx=0)
  (rnn): LSTM(20, 64, batch_first=True, bidirectional=True)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [47]:
import numpy as np
with open('1268-0.txt', 'r', encoding='utf8') as fp:
    text = fp.read()

In [48]:
start_indx = text.find('THE MYSTERIOUS ISLAND')
end_indx = text.find('End of the Project Gutenberg')
text = text[start_indx:end_indx]
char_set = set(text)

print(text[:5])

THE M


In [50]:
print(f'Total length: {len(text)}')
print(f'Unique Characters: {len(char_set)}')

Total length: 1112310
Unique Characters: 80


In [54]:
chars_sorted = sorted(char_set)
char2int = {ch:i for i, ch in enumerate(chars_sorted)}
char_array = np.array(chars_sorted)
text_encoded = np.array(
    [char2int[ch] for ch in text],
    dtype=np.int32
)

print(f'Text encode shape: {text_encoded.shape}')

print(text[:15], '== Encoding ==> ', text_encoded[:15])
print(f'{text_encoded[15:21]} == Reverse ==> {"".join(char_array[text_encoded[15:21]])}')





Text encode shape: (1112310,)
THE MYSTERIOUS  == Encoding ==>  [44 32 29  1 37 48 43 44 29 42 33 39 45 43  1]
[33 43 36 25 38 28] == Reverse ==> ISLAND


In [56]:
for ex in text_encoded[:10]:
    print(f'{ex} --> {char_array[ex]}')

44 --> T
32 --> H
29 --> E
1 -->  
37 --> M
48 --> Y
43 --> S
44 --> T
29 --> E
42 --> R


In [74]:
from torch.utils.data import Dataset, DataLoader
seq_length = 40
chunk_size = seq_length + 1
text_chunks = [text_encoded[i:i+chunk_size] for i in range(len(text_encoded) - chunk_size+1)]

class TextDataset(Dataset):

    def __init__(self, text_chuncks):
        super().__init__()
        self.text_chunks = text_chuncks

    def __len__(self):
        return len(self.text_chunks)
    
    def __getitem__(self, index):
        text_chunk = self.text_chunks[index]

        return text_chunk[:-1].long(), text_chunk[1:].long()
    


seq_dataset = TextDataset(torch.tensor(np.array(text_chunks)))



In [75]:
for i , (seq, target) in enumerate(seq_dataset):

    print(f' Input (x): {repr("".join(char_array[seq]))}')

    print(f' Target(y): {repr("".join(char_array[target]))}')

    print()

    if i == 1:
        break

 Input (x): 'THE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n1'
 Target(y): 'HE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n18'

 Input (x): 'HE MYSTERIOUS ISLAND\n\nby Jules Verne\n\n18'
 Target(y): 'E MYSTERIOUS ISLAND\n\nby Jules Verne\n\n187'



In [76]:
batch_size = 64
torch.manual_seed(1)

seq_dl = DataLoader(seq_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [77]:
class RNN(nn.Module):

    def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn_hidden_size = rnn_hidden_size

        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True)
        self.fc = nn.Linear(rnn_hidden_size, vocab_size)

    def forward(self, x, hidden, cell):
        out = self.embedding(x).unsqueeze(1)
        out, (hidden, cell) = self.rnn(out, (hidden, cell))
        out = self.fc(out).reshape(out.size(0), -1) 
        return out, hidden, cell


    def init_hidden(self, batch_size):
        hidden = torch.zeros(1, batch_size, self.rnn_hidden_size)
        cell = torch.zeros(1, batch_size, self.rnn_hidden_size)

        return hidden, cell   

In [78]:
vocab_size = len(char_array)
print(vocab_size)

embed_dim = 256
rnn_hidden_size = 512
model = RNN(vocab_size, embed_dim, rnn_hidden_size)

model

80


RNN(
  (embedding): Embedding(80, 256)
  (rnn): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=80, bias=True)
)

In [79]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)



In [80]:
num_epochs = 10000
torch.manual_seed(1)

for epoch in range(num_epochs):

    hidden, cell = model.init_hidden(batch_size)

    seq_batch, target_batch = next(iter(seq_dl))

    optimizer.zero_grad()

    loss = 0
    for c in range(seq_length):
        pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
        loss += loss_fn(pred, target_batch[:, c])
    loss.backward()
    optimizer.step()

    loss = loss.item()/seq_length

    if epoch % 500 == 0:
        print(f'Epoch {epoch} loss {loss:.4f}')

Epoch 0 loss 4.3712
Epoch 500 loss 1.4020
Epoch 1000 loss 1.3461
Epoch 1500 loss 1.2838
Epoch 2000 loss 1.1827
Epoch 2500 loss 1.1821
Epoch 3000 loss 1.1526
Epoch 3500 loss 1.1720
Epoch 4000 loss 1.1512
Epoch 4500 loss 1.1236
Epoch 5000 loss 1.1726
Epoch 5500 loss 1.1326
Epoch 6000 loss 1.1057
Epoch 6500 loss 1.1560
Epoch 7000 loss 1.1007
Epoch 7500 loss 1.1783
Epoch 8000 loss 1.1811
Epoch 8500 loss 1.1765
Epoch 9000 loss 1.1004
Epoch 9500 loss 1.1662


In [81]:
for c in range(seq_length):
    print(seq_batch[:, c])

    if c == 1:
        break
        
        

tensor([64, 51, 61, 58, 64, 69, 54, 63, 61, 61, 69, 69, 69,  1,  1, 67,  1, 58,
        63, 68, 68, 51,  9, 54, 58, 57, 77, 58, 58, 69,  1, 69, 64, 63, 61, 57,
        74,  1, 58, 54,  1, 68,  1, 55, 68, 52, 63,  0, 63,  1, 50, 50, 57, 57,
        68, 57,  7, 63, 63, 50, 79, 64,  0,  1])
tensor([55, 68, 74, 64, 63, 57, 53,  1, 54, 54, 67, 67, 67, 67, 54, 54, 50, 63,
        64,  0,  1,  7,  1,  1, 68, 54, 68, 53, 68,  1, 65, 64,  1, 54, 54, 54,
        68, 52, 64, 63, 44,  1, 51, 64, 57, 67,  9, 72, 53, 58,  1,  1, 50, 54,
        69, 50,  1, 53, 56, 69,  0, 72, 44, 69])


In [84]:
seq_batch.shape

torch.Size([64, 40])

In [93]:
from torch.distributions.categorical import Categorical

torch.manual_seed(1)
logits = torch.tensor([[1.0, 1.0, 1.0]])
print(f'Probabilities: {nn.functional.softmax(logits, dim=1).numpy()[0]}')
m = Categorical(logits=logits)

samples = m.sample((10,))

print(samples.numpy())

Probabilities: [0.33333334 0.33333334 0.33333334]
[[0]
 [0]
 [0]
 [0]
 [1]
 [0]
 [1]
 [2]
 [1]
 [1]]


In [94]:
torch.manual_seed(1)
logits = torch.tensor([[1.0, 2.0, 3.0]])

print(f'Probabilities: {nn.functional.softmax(logits, dim=1).numpy()}')

m = Categorical(logits=logits)

samples = m.sample((10,))
print(samples.numpy())

Probabilities: [[0.09003057 0.24472848 0.66524094]]
[[0]
 [1]
 [1]
 [1]
 [2]
 [1]
 [2]
 [2]
 [2]
 [2]]


In [97]:
def sample(model, starting_str, len_generated_text=500, scale_factor=1.0):

    encoded_input = torch.tensor(
        [char2int[s] for s in starting_str]
    )

    encoded_input = torch.reshape(
        encoded_input, (1, -1)
    )

    generated_str = starting_str

    model.eval()

    hidden, cell = model.init_hidden(1)
    for c in range(len(starting_str) -1):
        _, hidden, cell = model(
            encoded_input[:, c].view(1), hidden, cell
        )

    last_char = encoded_input[:, -1]
    for _ in range(len_generated_text):
        logits, hidden, cell = model(
            last_char.view(1), hidden, cell
        )

        logits = torch.squeeze(logits, 0)
        scaled_logits = logits * scale_factor
        m = Categorical(logits=scaled_logits)
        last_char = m.sample()
        generated_str += str(char_array[last_char.item()])

    return generated_str

In [98]:
torch.manual_seed(1)
print(sample(model, starting_str='The island'))

The island would do not be
slace the colonists had crossed enough carefully thought that he aboundon of the voyage flamm had been that his hand hard of dry wounds oyer. They became which the picious animation elselors, which
fell and family and Pencroft and his plate! The wood and holes to be
cross arrive spother of the other apparance was stoppment, and had returning her departure of the water.

Cyrus Harding, after you are themselves under sturfing,” added Ayrton since he thom cording, dastance them to 


In [99]:
torch.manual_seed(1)
print(sample(model, starting_str='The island', scale_factor=2.0))

The island was the man who was the storm had been accustomed to the same cart. They were stranded on the stranger fell into a vessel was that our reply was not been difficult.

“Well, a little corral!”

“But, at least the back of the balloon, who did not have the corral. The temperature was of a great distance to the coast of the colonists found the colonists of the
southwest way from the water.

Cyrus Harding, after the two oars was provisions and searched and with a sufficient forming and the same momen


In [100]:
torch.manual_seed(1)
print(sample(model, starting_str='The island', scale_factor=0.5))

The island
focket do’k over luch” esteemed irreddl colous, ends, fcafthw, a trovels fills, wait!
Over; the desaikffall Lix Bonvedn’s woult had qutrain only edbybulsy occylude.
Froase day, which, rifinel othermentileasiness “cord lying!” crie, “Why dresh Harding,
Bob Yoth Amerat,” replied, Romethwesh:--”

Thatzing wished our ‘Daw-Wus?!
Two,” answ Pencroft.?
exemors, as he had predaty Granite House, they followank, admiy, MalTabusaquakeff disles?”
fixed ridble stawnualized;, our mountain wadasteements Romem
